#  COLLABORATIVE-FILTERING

In [88]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity as cs
from sklearn.feature_extraction.text import TfidfVectorizer as tfv
import joblib

# INTRODUCTION

In [89]:
links = pd.read_csv("links.csv")
movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")
tags = pd.read_csv("tags.csv")

# Data Inspection

In [90]:
#links.head()
movies.tail()
ratings.tail(20)
#tags.head()

,userId,movieId,rating,timestamp
100816,610,158872,3.5,1493848024
100817,610,158956,3.0,1493848947
100818,610,159093,3.0,1493847704
100819,610,160080,3.0,1493848031
100820,610,160341,2.5,1479545749
100821,610,160527,4.5,1479544998
100822,610,160571,3.0,1493848537
100823,610,160836,3.0,1493844794
100824,610,161582,4.0,1493847759
100825,610,161634,4.0,1493848362


In [91]:
movies.isnull().sum()

,0
movieId,0
title,0
genres,0


# Important Dataset Combination

In [92]:
mr = pd.merge(ratings, movies, on='movieId')
# Reorder columns: title and genres after movieId
columns_order = ['userId', 'movieId', 'title', 'genres', 'rating', 'timestamp']
mr = mr[columns_order]
mr.head(10)

,userId,movieId,title,genres,rating,timestamp
0,1,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,4.0,964982703
1,1,3,Grumpier Old Men (1995),Comedy|Romance,4.0,964981247
2,1,6,Heat (1995),Action|Crime|Thriller,4.0,964982224
3,1,47,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,5.0,964983815
4,1,50,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,5.0,964982931
5,1,70,From Dusk Till Dawn (1996),Action|Comedy|Horror|Thriller,3.0,964982400
6,1,101,Bottle Rocket (1996),Adventure|Comedy|Crime|Romance,5.0,964980868
7,1,110,Braveheart (1995),Action|Drama|War,4.0,964982176
8,1,151,Rob Roy (1995),Action|Drama|Romance|War,5.0,964984041
9,1,157,Canadian Bacon (1995),Comedy|War,5.0,964984100


# Dataset Transformation to Pivot Table

In [93]:
pivot_table = mr.pivot_table(index='userId', columns='title', values='rating')
# Fill NaN values with 0 to represent no rating
pivot_table_filled = pivot_table.fillna(0)
display(pivot_table.head())

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [94]:
matrixT = pivot_table_filled.T
matrixT.head()

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
title,,,,,,,,,,,,,,,,,,,,,
'71 (2014),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0
'Hellboy': The Seeds of Creation (2004),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
'Round Midnight (1986),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
'Salem's Lot (2004),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
'Til There Was You (1997),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# Cosine Similarity

In [95]:
similarity = cs(matrixT)
similarity_df = pd.DataFrame(similarity, index=matrixT.index, columns=matrixT.index)
similarity_df.head()

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
title,,,,,,,,,,,,,,,,,,,,,
'71 (2014),1.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.141653,0.0,...,0.0,0.342055,0.543305,0.707107,0.0,0.0,0.139431,0.327327,0.0,0.0
'Hellboy': The Seeds of Creation (2004),0.0,1.000000,0.707107,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0
'Round Midnight (1986),0.0,0.707107,1.000000,0.000000,0.000000,0.0,0.176777,0.0,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0
'Salem's Lot (2004),0.0,0.000000,0.000000,1.000000,0.857493,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0
'Til There Was You (1997),0.0,0.000000,0.000000,0.857493,1.000000,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0


Why do we use cosine similarity?

-- focuses on pattern, not magnitude

In [96]:
movie_index = pd.Series(matrixT.index)
movie_index.head()

,title
0,'71 (2014)
1,'Hellboy': The Seeds of Creation (2004)
2,'Round Midnight (1986)
3,'Salem's Lot (2004)
4,'Til There Was You (1997)


# Recommend Function Creation

In [97]:
def recommend(movie_name):

    # get index of movie
    idx = movie_index[movie_index == movie_name].index[0]

    # similarity scores for that movie
    scores = list(enumerate(similarity[idx]))

    # sort by similarity (highest first)
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    # get top 5 similar movies (skip itself)
    top_movies = scores[1:6]

    recommended = []
    for i in top_movies:
        recommended.append(matrixT.index[i[0]])

    return recommended

# Testing Recommendation

In [98]:
recommend("Braveheart (1995)")

['Jurassic Park (1993)',
 'Terminator 2: Judgment Day (1991)',
 'Fugitive, The (1993)',
 'Forrest Gump (1994)',
 'Pulp Fiction (1994)']

In [100]:
joblib.dump(similarity, 'similarity.pkl')
joblib.dump(movie_index, "movies.pkl")

['movies.pkl']

# Findings
  In collaborative filtering recommendation, dataset need to be transform to pivot table for cosine similarity to be able to detect similarities across users preference movie.

# Conclusion
TF-IDF wasn't used here because it only works with text related column. Cosine similarity failed working with sparsed data so i had to convert all NaN to 0, Above all system performed accurately and it is a bit more complex than content-based filtering